In [1]:
"""
Build assumption_master sheet: admin_fees
-----
Extracts Admin (% of NET Premium) per product from the Admin table
at rows 4-17, columns G-H of the Assumptions sheet.

Schema: product | variant | admin_pct | source_label | source_row | notes
variant ∈ {default, accelerator, standalone, dc}

The Admin table is a flat product-to-percent lookup where variant is
encoded in the row label (e.g., "LSDB" vs "Standalone LSDB" vs "DC").
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME = "admin_fees"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")


def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)


def peek(ws, row, col, span=2):
    """Show cells to the right of (row, col)."""
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        parts.append(f"{get_col_letter(col + dc)}{row}={v!r}")
    return " | ".join(parts)


# =============================================================================
# DIAGNOSTIC — walk rows 4-17, columns G-H
# Confirm headers land where expected and each row has (label, pct)
# =============================================================================
print("=" * 90)
print("DIAGNOSTIC — Admin table anchors (G4:H17)")
print("=" * 90)

for row in range(4, 18):
    label = ws.cell(row=row, column=7).value   # col G
    value = ws.cell(row=row, column=8).value   # col H
    print(f"  G{row:2d}: label={label!r:35s}  H{row:2d}={value!r}")

# Verify header row
header_g = ws.cell(row=4, column=7).value
header_h = ws.cell(row=4, column=8).value
assert isinstance(header_g, str) and header_g.strip().lower() == "product", (
    f"G4 should be 'Product', got {header_g!r}"
)
assert isinstance(header_h, str) and "net premium" in header_h.strip().lower(), (
    f"H4 should mention 'NET Premium', got {header_h!r}"
)
print("\n✓ Headers verified: G4=Product, H4=% of NET Premium")


# =============================================================================
# Product mapping: source label → (canonical product key, variant)
# In-scope only. SCIC / Education loaded but marked out-of-scope in notes.
# =============================================================================
PRODUCT_MAP = {
    # source label              (product key,           variant)
    "GLA":                      ("gla",                 "default"),
    "SP Life":                  ("spouse_gla",          "default"),
    "CI":                       ("ci",                  "accelerator"),
    "LSDB":                     ("lsdb",                "accelerator"),
    "Dismemberment":            ("dismemberment",       "default"),
    "Standalone Critical Inc":  ("ci",                  "standalone"),
    "Acc Death":                ("accident_death",      "default"),
    "Acc DB":                   ("accident_disability", "default"),
    "SP DB":                    ("spouse_lsdb",         "default"),
    "Standalone LSDB":          ("lsdb",                "standalone"),
    "DC":                       ("__dc_row__",          "dc"),
    "SCIC":                     (None,                  None),   # out of scope
    "Education":                (None,                  None),   # out of scope
}

# =============================================================================
# Harvest rows 5-17, cols G (label), H (admin_pct)
# =============================================================================
BLOCK_ROW_START = 5
BLOCK_ROW_END   = 18   # exclusive; covers rows 5..17

all_rows: list[dict] = []
skipped: list[tuple[str, str]] = []

for row in range(BLOCK_ROW_START, BLOCK_ROW_END):
    source_label = ws.cell(row=row, column=7).value  # col G
    if source_label is None:
        continue
    source_label = str(source_label).strip()

    if source_label not in PRODUCT_MAP:
        raise RuntimeError(
            f"Unexpected product label at G{row}: {source_label!r}. "
            f"Add to PRODUCT_MAP or mark as skip."
        )

    canonical, variant = PRODUCT_MAP[source_label]
    if canonical is None:
        skipped.append((source_label, f"G{row}"))
        continue

    admin_val = ws.cell(row=row, column=8).value  # col H
    if admin_val is None:
        continue

    # Special handling: "DC" row applies to BOTH gla and lsdb.
    # Emit two parquet rows so lookup by (product, variant='dc') works.
    if canonical == "__dc_row__":
        for dc_product in ("gla", "lsdb"):
            all_rows.append({
                "product":      dc_product,
                "variant":      "dc",
                "admin":        admin_val,
                "source_label": source_label,
                "source_row":   row,
                "notes":        "Defined Contribution variant admin (shared DC row)",
            })
        continue

    all_rows.append({
        "product":      canonical,
        "variant":      variant,
        "admin":        admin_val,
        "source_label": source_label,
        "source_row":   row,
        "notes":        f"Admin (% of NET Premium) — {source_label}",
    })

# =============================================================================
# Synthetic rows — products not in Assumptions Admin table
# =============================================================================
# Terminal Illness Cover uses LSDB Admin fees.
# Source: TT convention — TI Admin is not a separate row in G5:H17,
# instead TT reuses the LSDB Admin value. On A2026_07 that is 0.075.
#
# We mirror LSDB's 'accelerator' variant as terminal's 'default' since
# that is LSDB's primary rate. If TT ever splits TI variants we can add
# more terminal rows here.

lsdb_accel_admin = next(
    (r["admin"] for r in all_rows
     if r["product"] == "lsdb" and r["variant"] == "accelerator"),
    None,
)
assert lsdb_accel_admin is not None, (
    "Cannot synthesize terminal row — LSDB accelerator row missing from harvest."
)

all_rows.append({
    "product":      "terminal",
    "variant":      "default",
    "admin":        lsdb_accel_admin,
    "source_label": "(synthetic — TI uses LSDB fees)",
    "source_row":   0,
    "notes":        "TT convention: Terminal Illness Cover admin mirrors LSDB admin",
})
# =============================================================================
# Classify types
# =============================================================================
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["admin"]
    r["param_type"] = classify(v)
    r["admin_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "product",
    "variant",
    pl.col("admin_str").alias("admin"),
    "param_type",
    "source_label",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
# Composite key (product, variant) must be unique
dupes = df.group_by(["product", "variant"]).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

# Expected products present
expected_products = {
    "gla", "lsdb", "ci",
    "spouse_gla", "spouse_lsdb","terminal",
    "accident_death", "accident_disability",
    "dismemberment",
}
actual_products = set(df["product"].unique().to_list())
missing = expected_products - actual_products
assert not missing, f"Missing products: {missing}"

# All values numeric
for row in df["param_type"].value_counts().iter_rows(named=True):
    assert row["param_type"] in ("float", "int"), (
        f"Non-numeric admin found: type={row['param_type']!r}"
    )

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_admin(product: str, variant: str) -> float:
    row = df.filter(
        (pl.col("product") == product) & (pl.col("variant") == variant)
    )
    assert row.height == 1, f"Missing: {product}/{variant}"
    return float(row["admin"][0])

frozen = [
    ("gla",                 "default",     0.06),
    ("gla",                 "dc",          0.06),
    ("lsdb",                "accelerator", 0.075),
    ("lsdb",                "standalone",  0.075),
    ("lsdb",                "dc",          0.06),
    ("ci",                  "accelerator", 0.075),
    ("ci",                  "standalone",  0.075),
    ("spouse_gla",          "default",     0.06),
    ("spouse_lsdb",         "default",     0.075),
    ("accident_death",      "default",     0.06),
    ("accident_disability", "default",     0.075),
    ("dismemberment",       "default",     0.075),
    ("terminal",            "default",     0.075),
]
for product, variant, expected in frozen:
    actual = get_admin(product, variant)
    assert abs(actual - expected) < 1e-9, (
        f"{product}/{variant}: got {actual}, expected {expected}"
    )

print(f"\n✓ {df.height} rows validated")
print(f"✓ {len(expected_products)} in-scope products present")
print(f"✓ {len(frozen)} frozen value checks passed")
print(f"\nSkipped (out of scope): {[s[0] for s in skipped]}")
print(f"\nFinal DataFrame:")
print(df)


# =============================================================================
# Write parquet + update master workbook
# =============================================================================
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"\n✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")


# =============================================================================
# Append to build_log.csv
# =============================================================================
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "Admin (% of NET Premium) per product, extracted from Assumptions "
        "sheet G4:H17. Variant encoded in row label: LSDB vs Standalone LSDB, "
        "CI vs Standalone Critical Inc, and shared 'DC' row applies to both "
        "gla and lsdb. SCIC and Education skipped (out of scope)."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

Building: admin_fees
Source:   Basis and Assumptions.xlsx > Assumptions
DIAGNOSTIC — Admin table anchors (G4:H17)
  G 4: label='Product'                            H 4='% of NET Premium'
  G 5: label='GLA'                                H 5=0.06
  G 6: label='SP Life'                            H 6=0.06
  G 7: label='CI'                                 H 7=0.075
  G 8: label='LSDB'                               H 8=0.075
  G 9: label='Dismemberment'                      H 9=0.075
  G10: label='Standalone Critical Inc'            H10=0.075
  G11: label='Acc Death'                          H11=0.06
  G12: label='Acc DB'                             H12=0.075
  G13: label='SP DB'                              H13=0.075
  G14: label='Standalone LSDB'                    H14=0.075
  G15: label='DC'                                 H15=0.06
  G16: label='SCIC'                               H16=0.075
  G17: label='Education'                          H17=0.075

✓ Headers verified: G4=Product, H4=%